In [74]:
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import JinaEmbeddings
from langchain_community.vectorstores import FAISS


In [75]:
load_dotenv()

True

In [76]:
groq_key = os.getenv("GROQ_API_KEY")
jina_key = os.getenv("JINA_API_KEY")

print("env were loaded")

env were loaded


In [77]:
DATA_FILE_PATH = os.path.join("data","Hr_policy.txt")

In [78]:
loader = TextLoader(DATA_FILE_PATH , encoding="utf-8")
documents = loader.load()

In [79]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

chunks = splitter.split_documents(documents)

In [80]:
embeddings_model = JinaEmbeddings(
    model_name="jina-embeddings-v2-base-en"
)

In [81]:
vector_store = FAISS.from_documents(chunks, embeddings_model)
print("chunks ara stored" , vector_store.index.ntotal)

chunks ara stored 37


In [82]:
retriever = vector_store.as_retriever(search_kwargs = {"k":3})

def search_hr_policy(question:str)->str:
    """search the hr policy document for information about leave, work from home,
    probation, notice period, reimbursement, code of conduct, holiday, or exitprocess"""

    matching_chunks = retriever.invoke(question)
    return"\n\n".join(chunk.page_content for chunk in matching_chunks)

In [88]:
from langchain_groq import ChatGroq
from langchain.agents import create_agent

llm = ChatGroq(
            model = "openai/gpt-oss-120b",
               temperature=0
               )
llm.model_name

'openai/gpt-oss-120b'

In [89]:
HR_assistant = create_agent (
    model = llm,
    tools = [search_hr_policy],
    system_prompt="""
    you are a friendly hr assistant.
    ALways use the search_hr_policy tool to look up
    facts before answering.
    If the answer isn't in the search result, say you don't know"
    instead of guessing"
    
    """
)

In [91]:

response = HR_assistant.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Tell me about the leave policy. How do I apply for leave?"
            }
        ]
    }
)

print(response["messages"][-1].content)



**Leave Policy Overview**

| Leave Type | When It Can Be Used | How Far in Advance You Should Request | What You Need to Do |
|------------|--------------------|---------------------------------------|---------------------|
| **Earned Leave** (planned personal time / vacation) | For any planned personal time off. | At least **7 calendar days** before the intended start date. | Submit a request in the company’s **designated leave‑management system** and wait for manager approval. |
| **Casual Leave** (personal matters or unexpected situations) | For short‑notice personal matters or emergencies. | As soon as possible – the sooner you inform, the better. | Inform your **reporting manager** promptly and then log the request in the leave‑management system. |
| **Sick Leave** (illness or medical reasons) | When you are unable to work due to health reasons. | Notify your manager **as soon as possible** (ideally before your shift starts). | Log the request in the leave‑management system. For e